# Anion-binding ESP calibration (illustrative reproduction)

This notebook sketches an ESP→log(Ka) correlation workflow using placeholder training data bundled with suprakit.

Literature anchor: Priyadarshini *et al.*, *JACS Au* **2025** — replace the DOI below with the published article DOI when citing formally.


In [ ]:
# Run with: make notebook
from suprakit.core.device import detect_device

print(detect_device())


In [ ]:
from suprakit.anionfit.receptors import build_resorcin4arene

codes = ["Br", "CHO", "NO2", "CN"]
mols = [build_resorcin4arene([c] * 4, upper_rim="hydroxyl") for c in codes]
len(mols)


## ESP values (GFN2-xTB probe)

Computed ESP uses **GFN2-xTB** Mulliken atomic charges via `tblite.ase.TBLite`, a classical **point-charge**
approximation at the bowl probe (see `docs/modules/anionfit.md`). This is real physics-based screening
electrostatics—not the placeholder column in `priyadarshini_2025.csv`.

If `tblite` is not installed (`uv sync --extra xtb`), the next cell raises `ImportError` with install hints.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from suprakit.anionfit.esp import PROBE_OFFSET_BOHR, compute_esp_batch
from suprakit.core.xtb import is_tblite_available

cache_path = Path("notebooks/.cache/esp_reference.csv")
cache_path.parent.mkdir(parents=True, exist_ok=True)

# `mols` and `codes` are defined in the previous cell.
if is_tblite_available() and "mols" in globals() and "codes" in globals():
    if os.environ.get("SUPRAKIT_RECOMPUTE_ESP") == "1" or not cache_path.is_file():
        results = compute_esp_batch(
            mols,
            solvent="thf",
            optimize=False,
            n_jobs=1,
        )
        rows = []
        for code, r in zip(codes, results, strict=True):
            if isinstance(r, Exception):
                raise r
            rows.append(
                {
                    "substituent": code,
                    "esp_au": r.esp_au,
                    "probe_offset_bohr": float(PROBE_OFFSET_BOHR),
                }
            )
        df_esp = pd.DataFrame(rows)
        df_esp.to_csv(cache_path, index=False)
    else:
        df_esp = pd.read_csv(cache_path)

    display(df_esp)
else:
    print(
        "Skipping ESP batch: tblite not available or run the molecule-building cell first. "
        "Install with `uv sync --extra chem --extra xtb` (Linux CI wheels). "
        "Set SUPRAKIT_RECOMPUTE_ESP=1 to rebuild cache."
    )


### Placeholder log(Ka) values

The bundled `priyadarshini_2025.csv` uses **placeholder** `esp_au` and `logKa_THF` so OLS/bootstrap tests stay deterministic. **Do not confuse** those CSV scalars with the computed ESP values from the previous cell—the ESP workflow is real; the fitted literature targets still need SI transcription in a follow-up task.

In [ ]:
from suprakit.anionfit.esp import compute_esp_at_bowl_center
from suprakit.anionfit.receptors import build_resorcin4arene
from suprakit.core.xtb import is_tblite_available

if is_tblite_available():
    _a = build_resorcin4arene(["NO2"] * 4, upper_rim="hydroxyl")
    _b = build_resorcin4arene(["CN"] * 4, upper_rim="hydroxyl")
    _base = {"solvent": "thf", "optimize": False, "random_seed": 99}
    dlo = (
        compute_esp_at_bowl_center(_a, probe_offset_bohr=3.5, **_base).esp_au
        - compute_esp_at_bowl_center(_b, probe_offset_bohr=3.5, **_base).esp_au
    )
    dhi = (
        compute_esp_at_bowl_center(_a, probe_offset_bohr=4.0, **_base).esp_au
        - compute_esp_at_bowl_center(_b, probe_offset_bohr=4.0, **_base).esp_au
    )
    assert dlo * dhi > 0.0
    print("✓ ranking stable (probe offset 3.5 vs 4.0 Bohr)")
else:
    print("Skip ranking check: tblite not available in this environment.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from suprakit.anionfit.predict import ESPLogKaModel

model = ESPLogKaModel.from_priyadarshini_2025()
esp = model.esp_train
truth = model.logka_train

grid = np.linspace(float(np.min(esp)), float(np.max(esp)), 200)
pred, lo, hi = model.predict_with_ci(grid)

plt.figure(figsize=(6, 4))
plt.scatter(esp, truth, label="training (placeholder)")
plt.plot(grid, pred, label="OLS")
plt.fill_between(grid, lo, hi, alpha=0.2)
plt.xlabel("ESP (a.u.) — CSV placeholders for OLS demo")
plt.ylabel("log10 Ka")
plt.legend()
plt.tight_layout()


In [ ]:
from suprakit.anionfit.substituents import EWG_SMILES, enumerate_patterns

patterns = enumerate_patterns(list(EWG_SMILES.keys()), symmetric_only=False)
len(patterns)


In [ ]:
import numpy as np
import pandas as pd
from importlib.resources import files
from IPython.display import display

from suprakit.anionfit.predict import ESPLogKaModel
from suprakit.anionfit.substituents import EWG_SMILES, enumerate_patterns

model = ESPLogKaModel.from_priyadarshini_2025()
patterns = enumerate_patterns(list(EWG_SMILES.keys()), symmetric_only=False)

csv_path = files("suprakit.anionfit.data") / "priyadarshini_2025.csv"
df = pd.read_csv(csv_path, comment="#")
lookup = dict(zip(df["substituent"], df["esp_au"]))

rows = []
for pat in patterns[:30]:
    esp_mean = float(np.mean([lookup[c] for c in pat]))
    rows.append(
        {
            "pattern": "+".join(pat),
            "esp_mean_placeholder": esp_mean,
            "logKa_pred": float(model.predict(np.array([esp_mean]))[0]),
        }
    )

ranked = pd.DataFrame(rows).sort_values("logKa_pred", ascending=False).head(10)
display(ranked)

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.bar(ranked["pattern"], ranked["logKa_pred"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("predicted log10 Ka (placeholder)")
plt.tight_layout()


## Limitations

- Linearity is a modeling choice, not a physical law.
- Implicit solvation, conformational ensembles, and probe-point definitions dominate uncertainty.
